# Step 2 - Preprocessing & Feature Engineering

This notebook performs **data cleaning**, **feature engineering**, and **preprocessing setup** 
for the Adult Income dataset.

**Goals:**
- Clean both train and test datasets using rules derived from the **training data**.
- Apply deterministic feature engineering to both datasets.
- Define and save preprocessing pipeline structure (unfitted).
- Save only the **essential artifacts** needed for modeling in Step 3.

---

In [ ]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Paths
ARTIFACTS = Path('./artifacts'); ARTIFACTS.mkdir(exist_ok=True, parents=True)
DATA = Path('./data')
print('Artifacts directory:', ARTIFACTS.resolve())
print('Data directory:', DATA.resolve())


## 1. Load raw train and test data

In [ ]:
cols = [
    'age','workclass','fnlwgt','education','education-num','marital-status',
    'occupation','relationship','race','sex','capital-gain','capital-loss',
    'hours-per-week','native-country','income'
]

df_train = pd.read_csv(DATA/'adult.data', header=None, names=cols, na_values='?', skipinitialspace=True)
df_test = pd.read_csv(DATA/'adult.test', skiprows=1, header=None, names=cols, na_values='?', skipinitialspace=True)

# Fix test labels ('>50K.' -> '>50K')
df_test['income'] = df_test['income'].astype(str).str.replace('.', '', regex=False)

print('Train shape:', df_train.shape, '| Test shape:', df_test.shape)
df_train.head()

## 2. Cleaning (train-driven, applied to both)

In [ ]:
# Drop duplicates and irrelevant column
df_train = df_train.drop_duplicates().copy()
df_test = df_test.drop_duplicates().copy()

for d in (df_train, df_test):
    if 'fnlwgt' in d.columns:
        d.drop(columns=['fnlwgt'], inplace=True)
    for c in d.select_dtypes(include='object').columns:
        d[c] = d[c].str.strip()

# Handle missing categorical values: use mode from train
cat_na_cols = ['workclass', 'occupation', 'native-country']
modes_train = {c: df_train[c].mode(dropna=True)[0] for c in cat_na_cols}
for c in cat_na_cols:
    df_train[c] = df_train[c].fillna(modes_train[c])
    df_test[c] = df_test[c].fillna(modes_train[c])

# Encode target
target_map = {'<=50K': 0, '>50K': 1}
df_train['income'] = df_train['income'].map(target_map)
df_test['income'] = df_test['income'].map(target_map)

print('Missing values after cleaning:')
print(df_train.isna().sum()[df_train.isna().sum() > 0])

## 3. Feature Engineering (applied deterministically to both)

In [ ]:
def add_features(df):
    d = df.copy()
    d['is_senior'] = (d['age'] >= 60).astype(int)
    for col in ['capital-gain','capital-loss']:
        d[f'{col}_log'] = np.log1p(d[col])
    d['education_per_age'] = d['education-num'] / d['age']
    d['work_gain_ratio'] = d['hours-per-week'] * (d['capital-gain_log'] + 1)
    d['is_married'] = d['marital-status'].str.contains('Married', na=False).astype(int)
    d['gender_role'] = d['sex'] + '_' + d['relationship']
    bins = [0,25,35,45,55,65,np.inf]
    labels = ['<25','25–35','35–45','45–55','55–65','65+']
    d['age_group'] = pd.cut(d['age'], bins=bins, labels=labels, right=False)
    return d

df_train_fe = add_features(df_train)
df_test_fe = add_features(df_test)

print('Train shape after FE:', df_train_fe.shape, '| Test shape after FE:', df_test_fe.shape)

## 4. Save cleaned + feature-engineered datasets

In [ ]:
df_train_fe.to_csv(ARTIFACTS/'adult_train_clean_fe.csv', index=False)
df_test_fe.to_csv(ARTIFACTS/'adult_test_clean_fe.csv', index=False)

print('Saved:')
print('- adult_train_clean_fe.csv')
print('- adult_test_clean_fe.csv')

## 5. Define numeric and categorical feature lists

In [ ]:
numeric_features = [
    'age','education-num','hours-per-week',
    'capital-gain_log','capital-loss_log',
    'education_per_age','work_gain_ratio'
]
categorical_features = [
    'workclass','marital-status','occupation','relationship','race','sex','native-country',
    'is_senior','is_married','gender_role','age_group'
]

joblib.dump({'numeric_features': numeric_features, 'categorical_features': categorical_features},
            ARTIFACTS/'feature_lists.joblib')
print('Saved feature_lists.joblib')

## 6. Build preprocessing pipeline (be fitted in Step 3)

In [ ]:
numeric_transformer = Pipeline([
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Save unfitted preprocessor structure
joblib.dump(preprocessor, ARTIFACTS/'preprocessor_def.joblib')
print('Saved preprocessor_def.joblib (unfitted structure).')

---
### Summary
- Cleaned and feature-engineered **train** and **test** datasets saved.
- Preprocessing pipeline **defined but not fitted**.
- Ready for **Step 3 (Modeling)**, where you will:
  1. Split the training data into train/validation.
  2. Fit the preprocessor on `X_train`.
  3. Transform train/validation/test.
  4. Train and evaluate your models.

**Artifacts created:**
- `adult_train_clean_fe.csv`
- `adult_test_clean_fe.csv`
- `feature_lists.joblib`
- `preprocessor_def.joblib`
